# **Laboratuvar: Keras Tuner ile Hiperparametre Ayarlaması**

Bu laboratuvar çalışmasında, Keras Tuner'ı nasıl kuracağınızı ve hiperparametre ayarlaması için ortamı nasıl hazırlayacağınızı öğreneceksiniz.

## Öğrenme hedefleri:
Bu laboratuvar çalışmasının sonunda şunları yapabileceksiniz:
- Keras Tuner'ı kurmak ve gerekli kütüphaneleri içe aktarmak
- MNIST veri setini yüklemek ve ön işleme tabi tutmak
- Model mimarisini yapılandırmak için hiperparametreleri kullanan bir model oluşturma fonksiyonu tanımlamak
- En iyi hiperparametre yapılandırmasını aramak için Keras Tuner'ı kurmak
- Aramadan en iyi hiperparametreleri almak ve bu optimize edilmiş değerlerle bir model oluşturmak

## Önkoşullar:
- Python programlamanın temel anlayışı
- Keras ve TensorFlow kurulu olması

### Alıştırma 1: Keras Tuner Kurulumu

Bu alıştırma, Keras Tuner'ı kullanmak için ilk kurulum adımlarını size gösterir. Kütüphaneyi kuracak, gerekli modülleri içe aktaracak ve hiperparametre ayarlaması için kullanılacak MNIST veri setini yükleyip ön işleme tabi tutacaksınız.

1. **Keras Tuner Kurulumu:**

- Keras Tuner'ı kurmak için pip kullanın.
2. **Gerekli Kütüphaneleri İçe Aktarma:**

- Keras Tuner, TensorFlow ve Keras modüllerini içe aktarın.
3. **MNIST Veri Setini Yükleme ve Ön İşleme:**

- MNIST veri setini yükleyin.

- Veri setini 255.0'a bölerek normalleştirin.

In [2]:
!pip install tensorflow==2.16.2
!pip install keras-tuner==1.4.7
!pip install numpy<2.0.0



/bin/bash: line 1: 2.0.0: No such file or directory


#### Açıklama:
Bu kod, pip kullanarak gerekli kütüphaneleri kurar.

- **TensorFlow**: Keras Tuner ile uyumluluğu sağlar.

- **Keras Tuner**: Bu laboratuvarda kullanılan sürüm.

- **Numpy**: Diğer kurulu paketlerle uyumluluğu sağlar.

In [15]:
import sys

# Increase recursion limit to prevent potential issues
sys.setrecursionlimit(100000)

#### Açıklama: `sys.setrecursionlimit` fonksiyonu, özyineleme sınırını artırmak için kullanılır; bu da, derin iç içe fonksiyonlara sahip karmaşık modeller çalıştırırken veya TensorFlow gibi belirli kütüphaneler kullanırken olası özyineleme hatalarını önlemeye yardımcı olur.

In [4]:
# Step 2: Import necessary libraries
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import os
import warnings

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 = all logs, 1 = filter out INFO, 2 = filter out INFO and WARNING, 3 = ERROR only



2026-05-10 11:01:05.457871: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-10 11:01:05.459024: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-10 11:01:05.463301: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-10 11:01:05.475391: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-10 11:01:05.500643: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

#### Açıklama

Bu kod gerekli kütüphaneleri içe aktarır:

- **`keras_tuner`**: Hiperparametre ayarlaması için kullanılır.

- **`Sequential`**: Keras'ta doğrusal katman yığını.

- **`Dense`**, **`Flatten`**: Yaygın Keras katmanları.

- **`mnist`**: Görüntü sınıflandırması için standart bir veri kümesi olan MNIST veri kümesi.

- **`Adam`**: Keras'ta bir optimizasyon algoritması.

In [5]:

# Step 3: Load and preprocess the MNIST dataset
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0

print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


#### Açıklama

Bu kod, MNIST veri setini yükler ve ön işleme tabi tutar:

- **`mnist.load_data()`**: Veri setini yükler ve eğitim ile doğrulama bölümlerini döndürür.

- **`x_train / 255.0`**: Piksel değerlerini 0 ile 1 arasında olacak şekilde normalleştirir.
- **`print(f'...')`**: Eğitim ve doğrulama veri setlerinin şekillerini görüntüler.

### Alıştırma 2: Hiperparametrelerle Model Tanımlama

Bu alıştırmada, yoğun bir katmandaki birim sayısını ve öğrenme oranını belirtmek için `HyperParameters` nesnesini kullanan bir model oluşturma fonksiyonu tanımlayacaksınız. Bu fonksiyon, hiperparametre ayarlamasına hazır derlenmiş bir Keras modeli döndürür.

**Model oluşturma fonksiyonu tanımlayın:**
- Giriş olarak bir `HyperParameters` nesnesi alan bir `build_model` fonksiyonu oluşturun.

- Yoğun bir katmandaki birim sayısını ve optimize edici için öğrenme oranını tanımlamak üzere `HyperParameters` nesnesini kullanın.

- Seyrek kategorik çapraz entropi kaybı ve Adam optimize edici ile modeli derleyin.

In [16]:
# Define a model-building function 

def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### Açıklama

Bu fonksiyon, hiperparametrelerle bir Keras modeli oluşturur ve derler:

- **`hp.Int('units', ...)`**: Yoğun katmandaki birim sayısını hiperparametre olarak tanımlar.

- **`hp.Float('learning_rate', ...)`**: Öğrenme oranını hiperparametre olarak tanımlar.

- **`model.compile()`**: Modeli Adam optimizasyon algoritması ve seyrek kategorik çapraz entropi kaybı ile derler.

### Alıştırma 3: Hiperparametre Arama Yapılandırması

Bu alıştırma, Keras Tuner'ı yapılandırma konusunda size rehberlik eder. Model oluşturma fonksiyonunu, optimizasyon hedefini, deneme sayısını ve sonuçların saklanacağı dizini belirterek bir `RandomSearch` tuner'ı oluşturursunuz. Arama alanı özeti, ayarlanan hiperparametrelerin genel bir görünümünü sağlar.

**RandomSearch Tuner Oluşturma:**
- Keras Tuner'dan `RandomSearch` sınıfını kullanın.

- Model oluşturma fonksiyonunu, optimizasyon hedefini (doğrulama doğruluğu), deneme sayısını ve sonuçların saklanacağı dizini belirtin.

In [17]:
# Create a RandomSearch Tuner 

tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space 
tuner.search_space_summary()

Reloading Tuner from my_dir/intro_to_kt/tuner0.json
Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


#### Açıklama

Bu kod, bir Keras Tuner `RandomSearch`'ü şu şekilde kurar:

- **`build_model`**: Model oluşturma fonksiyonu.

- **`objective='val_accuracy'`**: Optimize edilecek ölçüt (doğrulama doğruluğu).

- **`max_trials=10`**: Denenecek farklı hiperparametre konfigürasyonlarının maksimum sayısı.

- **`executions_per_trial=2`**: Her konfigürasyonun çalıştırılma sayısı.

- **`directory='my_dir'`**: Sonuçların kaydedileceği dizin.

- **`project_name='intro_to_kt'`**: Sonuçları düzenlemek için kullanılacak projenin adı.

Hiperparametre arama alanının özetini gösterir ve ayarlanan hiperparametrelerin genel bir görünümünü sağlar.

### Alıştırma 4: Hiperparametre Arama İşlemi

Bu alıştırmada, tuner'ın `search` yöntemini kullanarak hiperparametre arama işlemini gerçekleştireceksiniz. Eğitim ve doğrulama verilerini, epoch sayısı ile birlikte sağlayacaksınız. Arama tamamlandıktan sonra, sonuç özeti bulunan en iyi hiperparametre yapılandırmalarını gösterecektir.

**Aramayı Çalıştırın:**
- Tuner'ın `search` yöntemini kullanın.

- Eğitim verilerini, doğrulama verilerini ve epoch sayısını girin.

In [18]:
# Run the hyperparameter search 
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

# Display a summary of the results 
tuner.results_summary() 

Results summary
Results in my_dir/intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 08 summary
Hyperparameters:
units: 480
learning_rate: 0.0004580647448380267
Score: 0.9795500040054321

Trial 05 summary
Hyperparameters:
units: 384
learning_rate: 0.0003588613453833401
Score: 0.9787000119686127

Trial 07 summary
Hyperparameters:
units: 448
learning_rate: 0.003256343224504914
Score: 0.9757999777793884

Trial 01 summary
Hyperparameters:
units: 96
learning_rate: 0.0028156208463983876
Score: 0.974049985408783

Trial 09 summary
Hyperparameters:
units: 224
learning_rate: 0.004025830961764418
Score: 0.9736000001430511

Trial 04 summary
Hyperparameters:
units: 160
learning_rate: 0.004737198185425148
Score: 0.972900003194809

Trial 06 summary
Hyperparameters:
units: 64
learning_rate: 0.0007036121313527428
Score: 0.9691500067710876

Trial 00 summary
Hyperparameters:
units: 128
learning_rate: 0.007538484149179979
Score: 0.9690000116825104

Trial 03 summary
H

#### Açıklama

Bu komut hiperparametre aramasını çalıştırır:

- **`epochs=5`**: Her deneme 5 epoch boyunca eğitilir.

- **`validation_data=(x_val, y_val)`**: Arama sırasında modelin performansını değerlendirmek için kullanılan doğrulama verileri.

Arama tamamlandıktan sonra, bu komut arama sırasında bulunan en iyi hiperparametre yapılandırmalarının bir özetini görüntüler.

## Alıştırma 5: En İyi Hiperparametrelerin Analizi ve Kullanımı

Bu alıştırmada, arama sırasında bulunan en iyi hiperparametreleri alacak ve değerlerini yazdıracaksınız. Ardından, bu optimize edilmiş hiperparametrelerle bir model oluşturacak ve onu tam eğitim veri seti üzerinde eğiteceksiniz. Son olarak, seçilen hiperparametrelerle iyi performans gösterdiğinden emin olmak için modelin performansını test setinde değerlendireceksiniz.

**En iyi hiperparametreleri alma:**
- En iyi hiperparametreleri almak için `get_best_hyperparameters` yöntemini kullanın.

- Hiperparametreler için en uygun değerleri yazdırın.

**Modeli oluşturma ve eğitme:**
- En iyi hiperparametreleri kullanarak bir model oluşturun.

- Modeli tam eğitim veri seti üzerinde eğitin ve performansını test setinde değerlendirin.

In [21]:
# Step 1: Retrieve the best hyperparameters 

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""") 

# Step 2: Build and Train the Model with Best Hyperparameters 
model = tuner.hypermodel.build(best_hps) 
model.fit(x_train, y_train, epochs=10, validation_split=0.2) 

# Evaluate the model on the test set 
test_loss, test_acc = model.evaluate(x_val, y_val) 
print(f'Test accuracy: {test_acc}') 

 

The optimal number of units in the first dense layer is 480. 

The optimal learning rate for the optimizer is 0.0004580647448380267. 


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.9220 - loss: 0.2824 - val_accuracy: 0.9576 - val_loss: 0.1534
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.9651 - loss: 0.1215 - val_accuracy: 0.9684 - val_loss: 0.1066
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.9779 - loss: 0.0794 - val_accuracy: 0.9707 - val_loss: 0.0940
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.9837 - loss: 0.0569 - val_accuracy: 0.9732 - val_loss: 0.0892
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.9878 - loss: 0.0423 - val_accuracy: 0.9775 - val_loss: 0.0747
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.9918 - loss: 0.0301 - val_accuracy: 0.9773 - val_loss: 0.0778
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0

#### Açıklama

Bu kod, arama sırasında bulunan en iyi hiperparametreleri alır:

- **`get_best_hyperparameters(num_trials=1)`**: En iyi hiperparametre yapılandırmasını alır.

- **`print(f"...")`**: En iyi hiperparametreleri yazdırır.

- **`model.fit(...)`**: Modeli, %20'lik bir doğrulama bölümüyle tam eğitim verileri üzerinde eğitir.

- **`model.evaluate(...)`**: Modeli test (doğrulama) veri kümesi üzerinde değerlendirir ve modelin ne kadar iyi genelleme yaptığını gösteren doğruluk oranını yazdırır.

## Uygulama Alıştırmaları

### Alıştırma 1: Keras Tuner Kurulumu

#### Amaç:
Keras Tuner'ı nasıl kuracağınızı ve hiperparametre ayarlaması için ortamı nasıl hazırlayacağınızı öğrenmek.

#### Talimatlar:
1. Keras Tuner'ı kurun.

2. Gerekli kütüphaneleri içe aktarın.

3. MNIST veri setini yükleyin ve ön işleme tabi tutun.

In [22]:
 import keras_tuner as kt 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.optimizers import Adam 

# Step 3: Load and preprocess the MNIST data set 
(x_train, y_train), (x_val, y_val) = mnist.load_data() 
x_train, x_val = x_train / 255.0, x_val / 255.0 

# Print the shapes of the training and validation datasets
print(f'Training data shape: {x_train.shape}') 
print(f'Validation data shape: {x_val.shape}')

Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


<details>
    <summary>Click here for Solution</summary>

```python
!pip install keras-tuner 

# Step 2: Import necessary libraries 
import keras_tuner as kt 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.optimizers import Adam 

# Step 3: Load and preprocess the MNIST data set 
(x_train, y_train), (x_val, y_val) = mnist.load_data() 
x_train, x_val = x_train / 255.0, x_val / 255.0 

# Print the shapes of the training and validation datasets
print(f'Training data shape: {x_train.shape}') 
print(f'Validation data shape: {x_val.shape}')
```

</details>


### Alıştırma 2: Hiperparametrelerle Model Tanımlama

#### Amaç:
Model mimarisini yapılandırmak için hiperparametreler kullanan bir model oluşturma fonksiyonu tanımlayın.

#### Talimatlar:
1. Yoğun katmandaki birim sayısını ve öğrenme oranını belirtmek için `HyperParameters` nesnesini kullanan bir model oluşturma fonksiyonu tanımlayın.

2. Seyrek kategorik çapraz entropi kaybı ve Adam optimizasyon algoritması ile modeli derleyin.

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

# Step 1: Define a model-building function
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

<details>
    <summary>Click here for Solution</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

# Step 1: Define a model-building function
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model
```

</details>


### Alıştırma 3: Hiperparametre Arama Yapılandırması

#### Amaç:
En iyi hiperparametre yapılandırmasını aramak için Keras Tuner'ı kurmak.

#### Talimatlar:
1. Model oluşturma fonksiyonunu kullanarak bir `RandomSearch` tuner oluşturun.

2. Optimizasyon amacını, deneme sayısını ve sonuçların saklanacağı dizini belirtin.

In [24]:
import keras_tuner as kt

# Step 1: Create a RandomSearch Tuner
tuner = kt.RandomSearch(
    build_model,  # Ensure 'build_model' function is defined from previous code
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space
tuner.search_space_summary()

Reloading Tuner from my_dir/intro_to_kt/tuner0.json
Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


<details>
    <summary>Click here for Solution</summary>

```python
import keras_tuner as kt

# Step 1: Create a RandomSearch Tuner
tuner = kt.RandomSearch(
    build_model,  # Ensure 'build_model' function is defined from previous code
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space
tuner.search_space_summary()
```

</details>


### Alıştırma 4: Hiperparametre Arama İşlemini Çalıştırma

#### Amaç:
Hiperparametre arama işlemini çalıştırın ve sonuçların özetini görüntüleyin.

#### Talimatlar:
1. Ayarlayıcının `search` yöntemini kullanarak hiperparametre arama işlemini çalıştırın.

2. Eğitim verilerini, doğrulama verilerini ve epoch sayısını girin.

3. Sonuçların özetini görüntüleyin.

In [25]:
# Step 1: Run the hyperparameter search 

tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

 # Display a summary of the results 

tuner.results_summary()

Results summary
Results in my_dir/intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 08 summary
Hyperparameters:
units: 480
learning_rate: 0.0004580647448380267
Score: 0.9795500040054321

Trial 05 summary
Hyperparameters:
units: 384
learning_rate: 0.0003588613453833401
Score: 0.9787000119686127

Trial 07 summary
Hyperparameters:
units: 448
learning_rate: 0.003256343224504914
Score: 0.9757999777793884

Trial 01 summary
Hyperparameters:
units: 96
learning_rate: 0.0028156208463983876
Score: 0.974049985408783

Trial 09 summary
Hyperparameters:
units: 224
learning_rate: 0.004025830961764418
Score: 0.9736000001430511

Trial 04 summary
Hyperparameters:
units: 160
learning_rate: 0.004737198185425148
Score: 0.972900003194809

Trial 06 summary
Hyperparameters:
units: 64
learning_rate: 0.0007036121313527428
Score: 0.9691500067710876

Trial 00 summary
Hyperparameters:
units: 128
learning_rate: 0.007538484149179979
Score: 0.9690000116825104

Trial 03 summary
H

<details>
    <summary>Click here for Solution</summary>

```python
# Step 1: Run the hyperparameter search 

tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

 # Display a summary of the results 

tuner.results_summary()
```

</details>


### Alıştırma 5: En İyi Hiperparametrelerin Analizi ve Kullanımı

#### Amaç: Arama sonucundan en iyi hiperparametreleri elde etmek ve bu optimize edilmiş değerlerle bir model oluşturmak.

#### Talimatlar:
1. `get_best_hyperparameters` yöntemini kullanarak en iyi hiperparametreleri elde edin.

2. En iyi hiperparametreleri kullanarak bir model oluşturun.

3. Modeli tam eğitim veri seti üzerinde eğitin ve performansını doğrulama seti üzerinde değerlendirin.

In [26]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 

print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""") 

 # Step 2: Build and train the model with best hyperparameters 

model = tuner.hypermodel.build(best_hps) 

model.fit(x_train, y_train, epochs=10, validation_split=0.2) 

 # Evaluate the model on the validation set 

val_loss, val_acc = model.evaluate(x_val, y_val) 

print(f'Validation accuracy: {val_acc}')

 

The optimal number of units in the first dense layer is 480. 

The optimal learning rate for the optimizer is 0.0004580647448380267. 


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - accuracy: 0.9203 - loss: 0.2851 - val_accuracy: 0.9563 - val_loss: 0.1553
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.9647 - loss: 0.1227 - val_accuracy: 0.9673 - val_loss: 0.1114
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.9759 - loss: 0.0821 - val_accuracy: 0.9670 - val_loss: 0.1064
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.9827 - loss: 0.0578 - val_accuracy: 0.9740 - val_loss: 0.0865
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.9875 - loss: 0.0428 - val_accuracy: 0.9757 - val_loss: 0.0804
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.9911 - loss: 0.0316 - val_accuracy: 0.9775 - val_loss: 0.0743
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0